# CascadeDebug — Phase 7 GRPO (Colab) — **upload this file directly**

1. In Colab: **File → Upload notebook** and choose this `.ipynb` (or open it from a cloned `colab_phase7` folder).
2. **Runtime → Change runtime type → T4 GPU** (Pro: A100 is faster).
3. Run all cells top to bottom. The training script is loaded from **GitHub `main`**, not from the `.ipynb` alone (clone + reset cell is required first).

**Default `PROFILE = "submission"`** (best balance for ~3h and strong results on T4):

| Item | Value |
|------|--------|
| Model | Qwen2.5-**7B** 4-bit (Unsloth) |
| Training steps | **150** (GRPO optimizer **steps** — not “150 epochs” over the full dataset) |
| Max new tokens / completion | **256** (matches `train_gpu.py`) |
| GRPO group & grad accum | 2, **8** |
| Time | **about 2.5–3.5 h** on many T4s (if your earlier 7B+300 was ~6h, 150 steps ≈ **half** that) |

**Other profiles** in `train_grpo_colab.py`: `full` = 7B, **300** steps; `light` = 3B, 90 steps (~30 min).

**After training:** download `results/*.png`, `results/training_log.csv`, and optionally `results/lora_adapter/`.


In [ ]:
# GPU check
!nvidia-smi

In [ ]:
# Install (mergekit is required — TRL's GRPOTrainer imports it; ~1–2 min)
%pip install -q unsloth "trl>=0.12.0" mergekit datasets matplotlib numpy huggingface_hub
print("OK: dependencies installed.")

In [ ]:
import os
from getpass import getpass

# Optional: Hugging Face write token for uploading plots to your Space (repo results/)
tok = getpass("HF write token (or press Enter to skip): ").strip()
if tok:
    os.environ["HF_TOKEN"] = tok
    print("HF_TOKEN set (length:", len(tok), ")")
else:
    print("No HF token — results stay local under results/ only.")

In [ ]:
# Clone and sync to latest main (so `train_grpo_colab.py` is current — default submission = 7B, 150 steps)
import os, subprocess
REPO = "/content/cascadedebug"
CLONE = "https://github.com/sparshagra/cascadedebug.git"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", CLONE, REPO], check=True)
subprocess.run("git fetch origin && git reset --hard origin/main", shell=True, cwd=REPO, check=True)
subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO)
os.chdir(REPO)  # so %run and paths resolve
print("CWD set to", REPO)

In [ ]:
# Quick check: default submission profile in the file from GitHub (7B, 150 steps)
f = "/content/cascadedebug/training/train_grpo_colab.py"
t = open(f, encoding="utf-8").read()
assert 'PROFILE = "submission"' in t, "Set PROFILE in script or pull latest main from GitHub"
assert "elif PROFILE == \"submission\":" in t, "Outdated file layout — pull latest"
assert "MAX_STEPS = 150" in t, "Re-run the clone/reset cell (expect 150 steps in submission block)"
assert "Qwen2.5-7B" in t, "7B model expected in submission block"
print("OK: submission = 7B, 150 steps — you can start training in the next cell.")

In [ ]:
# Train (re-installs nothing; uses repo you just synced)
import os
os.chdir("/content/cascadedebug")
%run training/train_grpo_colab.py